In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split, GridSearchCV

## 1- Charger le dataset de courbes de sertissage

In [3]:
print("Chargement du dataset...")
df = pd.read_pickle("crimp_force_curves_dataset.pkl")

# On utilise les vraies colonnes du dataset Zenodo
X_raw = df['Force_curve_raw'].tolist()  # Les valeurs de la courbe brute
y = df['Main-label_encoded'].values     # L'étiquette encodée (ou Binary_label_encoded)

Chargement du dataset...


## 2. Extraction des caractéristiques (Feature Engineering)

In [4]:
print("Extraction des caractéristiques des courbes...")
features = []
for curve in X_raw:
    arr = np.array(curve)
    feat = [
        np.max(arr),       # Force maximale atteinte
        np.min(arr),       # Force minimale
        np.mean(arr),      # Force moyenne
        np.std(arr),       # Écart-type (variation du signal)
        np.trapz(arr),     # Aire sous la courbe (énergie de sertissage)
        np.argmax(arr)     # Position temporelle du pic de force
    ]
    features.append(feat)

X = np.array(features)

Extraction des caractéristiques des courbes...


## 3. Séparation en 3 ensembles : Train (60%), Validation (20%), Test (20%)

Étape A : On isole d'abord 20% pour le Test final, et 80% pour le bloc (Train + Validation)

In [5]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

Étape B : On divise le bloc de 80% en Train (75% de 80% = 60% du total) et Validation (25% de 80% = 20% du total)

In [6]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)

print(f"\n--- Répartition des données ---")
print(f"Ensemble d'entraînement : {len(X_train)} courbes")
print(f"Ensemble de validation  : {len(X_val)} courbes")
print(f"Ensemble de test        : {len(X_test)} courbes\n")


--- Répartition des données ---
Ensemble d'entraînement : 1463 courbes
Ensemble de validation  : 488 courbes
Ensemble de test        : 488 courbes



## 4. Optimisation des hyperparamètres (Grid Search) et Évaluation

1. Définir la grille des paramètres à tester

In [7]:
param_grid = {
    'n_estimators': [50, 100, 200],       # Nombre d'arbres
    'max_depth': [10, 20, None],          # Profondeur maximale des arbres
    'min_samples_split': [2, 5, 10]       # Nombre min d'échantillons pour diviser un nœud
}
print("Recherche des meilleurs paramètres en cours...")
rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)

Recherche des meilleurs paramètres en cours...


2. Utilisation de GridSearchCV (qui gère l'entraînement et la validation croisée)

In [8]:
grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    cv=5,               # Validation croisée en 5 folds sur le bloc d'entraînement/validation
    scoring='accuracy', 
    n_jobs=-1
)
# On entraîne sur X_train_val (les 80% réservés pour l'apprentissage/validation)
grid_search.fit(X_train_val, y_train_val)

print("\nMeilleurs paramètres trouvés :")
print(grid_search.best_params_)


Meilleurs paramètres trouvés :
{'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 50}


3. Récupérer le meilleur modèle optimisé

In [9]:
best_model = grid_search.best_estimator_

4. best_model = grid_search.best_estimator_

In [10]:
y_test_pred = best_model.predict(X_test)
acc_test = accuracy_score(y_test, y_test_pred)
print(f"\n[Test] Précision finale avec le modèle optimisé : {acc_test:.4f}\n")

print("Rapport de classification détaillé (sur le Test) :")
print(classification_report(y_test, y_test_pred))


[Test] Précision finale avec le modèle optimisé : 0.9508

Rapport de classification détaillé (sur le Test) :
              precision    recall  f1-score   support

           0       0.96      0.97      0.96       326
           1       0.92      0.93      0.93       103
           2       0.93      0.90      0.91        59

    accuracy                           0.95       488
   macro avg       0.94      0.93      0.94       488
weighted avg       0.95      0.95      0.95       488



## 5. Sauvegarde du modèle entraîné

In [11]:
model_filename = "crimp_best_random_forest_model.pkl"
joblib.dump(best_model, model_filename)
print(f"Modèle optimisé sauvegardé avec succès sous : {model_filename}")

Modèle optimisé sauvegardé avec succès sous : crimp_best_random_forest_model.pkl
